# Dịch title Cado bằng Qwen3-14B-AWQ (4-bit AWQ) trên Kaggle

Bản Kaggle tách biệt từ v6. Giữ nguyên model **`Qwen/Qwen3-14B-AWQ`**, sampling, prompt, bộ QA và quy tắc nhập cache v2; chỉ thay cách tìm đầu vào/lưu checkpoint để phù hợp Kaggle.

- Bản dịch chạy local trên CUDA GPU, không gọi API dịch và không cần HF token.
- Cần đính kèm private Dataset chứa bundle v6, cache v2 và Parquet nguồn; thêm dependency Dataset `nadsa091/qwen3-14b-awq-offline-wheelhouse` và model Dataset `raufrahim/qwen3-14b-awq`. Package, dữ liệu và model đều được nạp từ Kaggle Input; không cần tải chúng qua Internet.
- Cache v2 được đọc chỉ-đọc; bản hoàn thành/không lặp được nhập vào cache v6 riêng. Cache gốc không bị sửa.
- Run/checkpoint nằm trong `/kaggle/working/title_translation_cado_qwen3_14b_awq_full_v3_sampled`; bản mirror được cập nhật mỗi 100 title như v6.
- Notebook từ chối GPU dưới compute capability 7.5 hoặc dưới 14 GiB VRAM. Kaggle có thể cấp P100; nếu vậy, cell GPU dừng trước khi tải model/dịch.
- Kaggle không đồng bộ Google Drive và không nhận hot-reload qua `qwen_control.json`. Dùng Kaggle Files persistence hoặc ZIP checkpoint xuất ra để tiếp tục phiên sau.

`semantic_verified` vẫn là `False`: QA kỹ thuật không thay thế đánh giá đúng nghĩa.


## Chuẩn bị Kaggle

1. Tải notebook này về máy, vào Kaggle → **Create → New Notebook** → **File → Import Notebook**, rồi chọn file `.ipynb`.
2. Đính kèm private Dataset chứa bundle v6, cache v2 và Parquet nguồn; thêm dependency Datasets `nadsa091/qwen3-14b-awq-offline-wheelhouse`, `nadsa091/qwen3-14b-awq-offline-sdists` và model Dataset `raufrahim/qwen3-14b-awq` làm Input. Notebook xác minh checksum dữ liệu/model rồi nạp cả hai từ Kaggle Input.
3. Trong **Settings / Session options**, chọn **Accelerator → GPU**. Internet có thể tắt; package, Parquet và model đều có sẵn trong Input. Nếu Kaggle cấp P100, cell kiểm tra sẽ dừng; cần phiên có T4/T4x2 hoặc GPU tương thích khác.
4. Nếu giao diện có tùy chọn **Persistence**, chọn **Files only** trước khi chạy để giữ `/kaggle/working` qua lần kết nối lại. Nếu không có, cần xuất/tải ZIP checkpoint trước khi phiên mất.
5. Bấm **Start Session**, rồi chạy **Run All** trong phiên tương tác. Không dùng **Save Version → Save & Run All** để tiếp tục một checkpoint đang chạy: Kaggle mở một phiên sạch riêng cho thao tác đó.

Giữ Dataset ở chế độ **Private**: SQLite checkpoint chứa nội dung bản dịch. Không tải checkpoint lên Dataset công khai.


In [ ]:
"""Small, privacy-safe notebook heartbeat; safe to embed before notebook cells."""
import contextlib as _health_contextlib
import json as _health_json
import os as _health_os
import re as _health_re
import subprocess as _health_subprocess
import threading as _health_threading
import time as _health_time
from datetime import datetime as _HealthDateTime, timezone as _HealthTimezone
from pathlib import Path as _HealthPath


class QwenHealthMonitor:
    def __init__(self, path, interval=30, clock=None):
        self.path = _HealthPath(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.clock = clock or _health_time.monotonic
        self.interval = interval
        self.lock = _health_threading.RLock()
        self.stop_event = _health_threading.Event()
        self.started = self.clock()
        self.last_progress = self.started
        self.last_emit = -float('inf')
        self.last_signature = None
        self.payload = {'phase': 'initializing', 'cell': None, 'state': 'starting'}
        self.worker = _health_threading.Thread(target=self._run, daemon=True)
        self.worker.start()

    def set_phase(self, phase, cell=None):
        if not isinstance(phase, str) or not _health_re.fullmatch(r'[a-z_]{1,48}', phase):
            phase = 'unknown'
        with self.lock:
            self.payload.update(phase=phase, state='running')
            if cell is not None:
                self.payload['cell'] = int(cell)
            self.last_progress = self.clock()
            self.last_signature = None
            for key in ('progress_done', 'progress_total', 'progress_unit'):
                self.payload.pop(key, None)
        self.emit(force=True)

    def progress(self, progress=None, **kwargs):
        values = dict(progress or {})
        values.update(kwargs)
        allowed = ('processed_this_session', 'completed_this_session', 'errors_this_session',
                   'total_this_session', 'completed_before_session', 'progress_done', 'progress_total')
        with self.lock:
            phase = values.get('phase')
            if isinstance(phase, str) and _health_re.fullmatch(r'[a-z_]{1,48}', phase):
                self.payload['phase'] = phase
            for key in allowed:
                value = values.get(key)
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    self.payload[key] = max(0, int(value))
            unit = values.get('progress_unit')
            if unit in ('bytes', 'installer_lines', 'titles', 'files'):
                self.payload['progress_unit'] = unit
            signature = (self.payload['phase'],) + tuple(self.payload.get(k) for k in allowed)
            if signature != self.last_signature:
                self.last_progress = self.clock()
                self.last_signature = signature
        self.emit()

    def installer_activity(self):
        with self.lock:
            done = self.payload.get('progress_done', 0) + 1
        self.progress(progress_done=done, progress_unit='installer_lines')

    def cell_finished(self, final=False):
        with self.lock:
            self.payload['state'] = 'completed' if final else 'waiting_next_cell'
            self.last_progress = self.clock()
        self.emit(force=True)
        if final:
            self.stop_event.set()

    def fail(self, error):
        with self.lock:
            self.payload.update(state='error', last_error_class=type(error).__name__)
        self.emit(force=True)
        self.stop_event.set()

    def snapshot(self):
        with self.lock:
            now = self.clock()
            age = max(0, now - self.last_progress)
            payload = dict(self.payload)
            terminal = payload['state'] in ('completed', 'error')
            threshold = {'install_dependencies': 900, 'verify_model': 600,
                         'loading_model': 600, 'generation': 300}.get(payload['phase'], 600)
            if not terminal and payload['state'] != 'waiting_next_cell':
                payload['state'] = 'suspected_stall' if age > threshold else 'running'
            payload.update(heartbeat_at_utc=_HealthDateTime.now(_HealthTimezone.utc).isoformat(),
                           last_progress_at_utc=_HealthDateTime.fromtimestamp(
                               _health_time.time() - age, _HealthTimezone.utc).isoformat(),
                           progress_age_seconds=round(age, 1), elapsed_seconds=round(now-self.started, 1),
                           stall_threshold_seconds=threshold, process_id=_health_os.getpid(),
                           process_cpu_seconds=round(sum(_health_os.times()[:2]), 2))
            checkpoint = self.path.parent / 'cache_qwen3_14b_awq.sqlite3'
            try:
                stat = checkpoint.stat()
                payload.update(checkpoint_bytes=stat.st_size,
                               checkpoint_age_seconds=round(max(0, _health_time.time()-stat.st_mtime), 1))
            except OSError:
                payload['checkpoint_available'] = False
            return payload

    def emit(self, force=False, gpu=False):
        with self.lock:
            if not force and self.clock()-self.last_emit < 10:
                return
            payload = self.snapshot()
            if gpu:
                try:
                    result = _health_subprocess.run(
                        ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used', '--format=csv,noheader,nounits'],
                        capture_output=True, text=True, timeout=3)
                    row = result.stdout.splitlines()[0].split(',')
                    payload.update(gpu_utilization_percent=int(row[0].strip()), gpu_memory_used_mib=int(row[1].strip()))
                except Exception:
                    payload['gpu_metrics_available'] = False
            temporary = self.path.with_suffix('.json.tmp')
            try:
                temporary.write_text(_health_json.dumps(payload, ensure_ascii=True), encoding='utf-8')
                temporary.replace(self.path)
            except Exception as exc:
                payload['health_write_error_class'] = type(exc).__name__
            print('[qwen-health] ' + _health_json.dumps(payload, ensure_ascii=True), flush=True)
            self.last_emit = self.clock()

    @_health_contextlib.contextmanager
    def stage(self, phase, cell=None, final=False):
        self.set_phase(phase, cell)
        try:
            yield
        except BaseException as exc:
            with self.lock:
                self.payload.update(state='error', last_error_class=type(exc).__name__)
            self.emit(force=True)
            self.stop_event.set()
            raise
        else:
            with self.lock:
                self.payload['state'] = 'completed' if final else 'waiting_next_cell'
                self.last_progress = self.clock()
            self.emit(force=True)
            if final:
                self.stop_event.set()

    def _run(self):
        while not self.stop_event.wait(self.interval):
            self.emit(force=True, gpu=True)

    def close(self):
        self.stop_event.set()

_qwen_health = QwenHealthMonitor(_HealthPath('/kaggle/working/title_translation_cado_qwen3_14b_awq_full_v3_sampled/qwen_health.json'))
def _qwen_health_post_run(result):
    error = getattr(result, 'error_before_exec', None) or getattr(result, 'error_in_exec', None)
    if error is not None:
        _qwen_health.fail(error)
_health_ip = __import__('IPython').get_ipython()
if _health_ip is not None:
    _previous_health_hook = getattr(_health_ip, '_qwen_health_post_run', None)
    if _previous_health_hook is not None:
        try:
            _health_ip.events.unregister('post_run_cell', _previous_health_hook)
        except Exception:
            pass
    _health_ip.events.register('post_run_cell', _qwen_health_post_run)
    _health_ip._qwen_health_post_run = _qwen_health_post_run

_qwen_health.set_phase('prepare_inputs', cell=2)
# Kaggle has read-only inputs and writable outputs; never mount or change Google Drive here.
import json
import re
import shutil
import zipfile
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from IPython import get_ipython

# Check the actual runtime before dependency installation or model-file hashing.
_qwen_health.set_phase('preflight_gpu', cell=2)
import torch
_gpu_probe = {
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'nvidia_device_present': Path('/dev/nvidia0').exists(),
    'nvidia_smi_available': bool(shutil.which('nvidia-smi')),
}
print('[qwen-boot] ' + json.dumps(_gpu_probe), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle chưa cấp CUDA GPU. Vào Edit notebook, Settings / Session options, chọn GPU T4 x2 rồi bắt đầu lại session; không cài dependency trên runtime CPU.')
_gpu_properties = torch.cuda.get_device_properties(0)
if _gpu_properties.total_memory < 14 * 1024**3 or (_gpu_properties.major, _gpu_properties.minor) < (7, 5):
    raise RuntimeError('GPU hiện tại không đủ VRAM hoặc compute capability để chạy Qwen3-14B-AWQ ExLlamaV2.')

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
KAGGLE_WORK_ROOT = Path('/kaggle/working')
WORKDIR = KAGGLE_WORK_ROOT / 'title_translation_cado_qwen3_14b_awq_full_v3_sampled'
WORKDIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR = Path('/kaggle/temp/title_translation_cado_qwen3_14b_awq_full_v3_sampled')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE = LOCAL_DIR / 'cache_qwen3_14b_awq.sqlite3'
WORKING_CACHE = WORKDIR / 'cache_qwen3_14b_awq.sqlite3'
BUNDLE_NAME = 'title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip'
LEGACY_RUN_DIR_NAME = 'title_translation_cado_qwen3_14b_awq_full_v2'
CHECKPOINT_ZIP_NAME = 'qwen3_14b_awq_kaggle_checkpoint.zip'

if not KAGGLE_INPUT_ROOT.exists():
    raise RuntimeError('Không thấy /kaggle/input. Hãy mở notebook trong Kaggle và đính kèm Kaggle Dataset đầu vào.')

bundle_candidates = [
    path for path in KAGGLE_INPUT_ROOT.rglob(BUNDLE_NAME)
    if path.is_file()
]
if len(bundle_candidates) == 1:
    BUNDLE_INPUT = bundle_candidates[0]
elif len(bundle_candidates) == 0:
    # Kaggle may recursively unpack both the outer Dataset ZIP and this
    # nested bundle ZIP. In that case the bundle is an extracted directory,
    # not a file with BUNDLE_NAME. Repack only its two known source files in
    # the same flat layout expected by the rest of this setup cell.
    bundle_dir_names = {BUNDLE_NAME, BUNDLE_NAME[:-4]}
    unpacked_bundle_dirs = [
        path for path in KAGGLE_INPUT_ROOT.rglob('*')
        if path.is_dir() and path.name in bundle_dir_names
    ]
    if len(unpacked_bundle_dirs) != 1:
        raise FileNotFoundError(
            'Cần đúng 1 bundle v6 trong Kaggle Input; '
            f'tìm thấy {len(bundle_candidates)} file ZIP và '
            f'{len(unpacked_bundle_dirs)} thư mục bundle đã giải nén. '
            'Hãy đính kèm private dataset theo guide.'
        )
    expected_bundle_files = (
        'title_translation_hybrid.py',
        'title_translation_pipeline.py',
    )
    unpacked_bundle_dir = unpacked_bundle_dirs[0]
    unpacked_files = {}
    for name in expected_bundle_files:
        matches = [path for path in unpacked_bundle_dir.rglob(name) if path.is_file()]
        if len(matches) != 1:
            raise FileNotFoundError(
                f'Thư mục bundle đã bung phải có đúng một {name}; '
                f'tìm thấy {len(matches)}.'
            )
        unpacked_files[name] = matches[0]
    repacked_bundle = LOCAL_DIR / BUNDLE_NAME
    with zipfile.ZipFile(repacked_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for name, path in unpacked_files.items():
            archive.write(path, arcname=name)
    BUNDLE_INPUT = repacked_bundle
    print('Kaggle đã tự bung bundle; đã đóng gói lại hai file nguồn:', unpacked_bundle_dir)
else:
    raise FileNotFoundError(
        f'Cần đúng 1 bundle v6 trong Kaggle Input; hiện tìm thấy {len(bundle_candidates)} file ZIP. '
        'Hãy chỉ đính kèm một private dataset theo guide.'
    )
LEGACY_CACHE_NAME = 'cache_qwen3_14b_awq.sqlite3'
legacy_cache_candidates = [
    path for path in KAGGLE_INPUT_ROOT.rglob(LEGACY_CACHE_NAME)
    if path.parent.name == LEGACY_RUN_DIR_NAME
    and (path.parent / 'hybrid_environment_lock.json').is_file()
]
if len(legacy_cache_candidates) > 1:
    raise RuntimeError('Tìm thấy nhiều checkpoint v2; hãy chỉ đính kèm một bản checkpoint nguồn.')
LEGACY_WORKDIR = legacy_cache_candidates[0].parent if legacy_cache_candidates else None

# A checkpoint ZIP exported by this notebook can restore a v6 run when Kaggle
# Files persistence is unavailable. Only flat, known runtime files are allowed.
allowed_snapshot_files = {
    'cache_qwen3_14b_awq.sqlite3', 'hybrid_environment_lock.json',
    'hybrid_run_state.json', 'legacy_cache_import.json',
    'qwen3_14b_awq_kaggle_package_lock.json', 'qwen_live_status.json', 'qwen_health.json',
}
snapshot_candidates = list(KAGGLE_INPUT_ROOT.rglob(CHECKPOINT_ZIP_NAME))
if len(snapshot_candidates) > 1:
    raise RuntimeError('Tìm thấy nhiều ZIP checkpoint Kaggle; chỉ đính kèm bản mới nhất.')
if snapshot_candidates:
    with zipfile.ZipFile(snapshot_candidates[0]) as archive:
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if path.is_absolute() or len(path.parts) != 1 or '..' in path.parts:
                raise RuntimeError('ZIP checkpoint có đường dẫn không an toàn; không giải nén.')
            if path.name not in allowed_snapshot_files:
                raise RuntimeError(f'Tệp không được phép trong ZIP checkpoint: {path.name}')
            destination = WORKDIR / path.name
            if not destination.exists() and not member.is_dir():
                destination.write_bytes(archive.read(member))
    print('Đã khôi phục snapshot checkpoint Kaggle vào:', WORKDIR)

BUNDLE_PATH = WORKDIR / BUNDLE_NAME
if BUNDLE_PATH.exists():
    import hashlib
    old_sha = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
    input_sha = hashlib.sha256(BUNDLE_INPUT.read_bytes()).hexdigest()
    has_prior_run = WORKING_CACHE.exists() or (WORKDIR / 'hybrid_run_state.json').exists()
    if old_sha != input_sha and has_prior_run:
        raise RuntimeError(
            'Bundle Kaggle khác với bundle đã dùng để tạo checkpoint. '
            'Không tự thay code khi resume; hãy dừng và yêu cầu kiểm tra trước.'
        )
    if old_sha != input_sha:
        shutil.copy2(BUNDLE_INPUT, BUNDLE_PATH)
else:
    shutil.copy2(BUNDLE_INPUT, BUNDLE_PATH)

if WORKING_CACHE.exists() and not LOCAL_CACHE.exists():
    shutil.copy2(WORKING_CACHE, LOCAL_CACHE)
    print('Đã khôi phục cache v6 từ /kaggle/working.')

has_v6_checkpoint = WORKING_CACHE.exists() or (WORKDIR / 'legacy_cache_import.json').exists()
if not has_v6_checkpoint and LEGACY_WORKDIR is None:
    raise FileNotFoundError(
        'Thiếu cache nguồn v2. Đính kèm private Kaggle Dataset có '
        'title_translation_cado_qwen3_14b_awq_full_v2/cache_qwen3_14b_awq.sqlite3 '
        'và hybrid_environment_lock.json; xem guide đi kèm.'
    )

if LEGACY_WORKDIR is not None:
    print('Đã tìm thấy checkpoint v2 để nhập chỉ-đọc:', LEGACY_WORKDIR)
else:
    print('Tiếp tục checkpoint v6 đã có; không cần nhập lại cache v2.')
print('Kaggle run folder:', WORKDIR)
print('Cache local runtime:', LOCAL_CACHE)

# Write privacy-safe status locally; there is no Google Drive mount or remote
# code-reload channel in the Kaggle port.
import traceback
_qwen_status_path = WORKDIR / 'qwen_live_status.json'
def _qwen_redact(value):
    value = str(value)
    value = re.sub(r'hf_[A-Za-z0-9]{16,}', '[REDACTED_HF_TOKEN]', value)
    value = re.sub(r'AIza[0-9A-Za-z_-]{20,}', '[REDACTED_GOOGLE_KEY]', value)
    value = re.sub(r'AQ\.[A-Za-z0-9_-]{20,}', '[REDACTED_API_KEY]', value)
    return value
def _qwen_kaggle_error_hook(result):
    error = getattr(result, 'error_before_exec', None) or getattr(result, 'error_in_exec', None)
    if error is None:
        raw_cell = getattr(getattr(result, 'info', None), 'raw_cell', '') or ''
        if '_qwen_full_run_cell_marker' not in raw_cell:
            return
        summary = globals().get('full_result')
        if not isinstance(summary, dict):
            return
        try:
            payload = {
                'state': summary.get('status', 'unknown'),
                'phase': 'run_cell_finished',
                'updated_at_utc': datetime.now(timezone.utc).isoformat(),
                'completed_unique_titles': summary.get('completed_unique_titles'),
                'total_unique_titles': summary.get('total_unique_titles'),
                'remaining_unique_titles': summary.get('remaining_unique_titles'),
                'generation': summary.get('generation'),
            }
            temp = _qwen_status_path.with_suffix('.json.tmp')
            temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
            temp.replace(_qwen_status_path)
        except Exception as log_error:
            print(f'[qwen-kaggle-observer] status write failed: {type(log_error).__name__}', flush=True)
        return
    try:
        payload = {
            'state': 'error',
            'phase': 'notebook_cell_error',
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'cell_execution_count': getattr(result, 'execution_count', None),
            'last_error_class': type(error).__name__,
        }
        temp = _qwen_status_path.with_suffix('.json.tmp')
        temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
        temp.replace(_qwen_status_path)
    except Exception as log_error:
        print(f'[qwen-kaggle-observer] status write failed: {type(log_error).__name__}', flush=True)
_qwen_ip = get_ipython()
if _qwen_ip is not None:
    _old_hook = getattr(_qwen_ip, '_qwen_kaggle_error_hook', None)
    if _old_hook is not None:
        try: _qwen_ip.events.unregister('post_run_cell', _old_hook)
        except Exception: pass
    _qwen_ip.events.register('post_run_cell', _qwen_kaggle_error_hook)
    _qwen_ip._qwen_kaggle_error_hook = _qwen_kaggle_error_hook

_qwen_health.cell_finished(final=False)


In [ ]:

_qwen_health.set_phase('install_dependencies', cell=3)
# Cài thư viện Transformers cho AWQ; giữ nguyên PyTorch/CUDA Kaggle đang có.
import importlib
import importlib.metadata
import importlib.util
import json
import subprocess
import sys
from packaging.version import Version

PACKAGE_LOCK = WORKDIR / 'qwen3_14b_awq_kaggle_package_lock.json'
FAST_TRANSFORMERS_VERSION = '5.20.2'
MIN_SUPPORTED_TRANSFORMERS = Version('5.17.0')
PACKAGE_NAMES = ['transformers', 'accelerate', 'gptqmodel', 'huggingface_hub', 'sacrebleu', 'pyarrow', 'pandas', 'numpy']
PACKAGE_SPECS = [
    f'transformers=={FAST_TRANSFORMERS_VERSION}', 'accelerate>=1.2', 'gptqmodel',
    'huggingface_hub>=0.30', 'sacrebleu', 'pyarrow>=18', 'pandas>=2.2,<3',
]

def installed_package_versions():
    versions = {}
    for name in PACKAGE_NAMES:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            # A persisted lock can outlive a Kaggle runtime reset; represent absent
            # packages as mismatches so the locked versions are installed below.
            versions[name] = None
    versions['torch'] = Version(importlib.metadata.version('torch')).base_version  # ignore +cpu/+cuXXX build tag
    versions['numpy'] = importlib.metadata.version('numpy')  # preserve Kaggle ABI
    return versions

_DNS_BOOTSTRAP_SOURCE = r"""
import http.client
import ipaddress
import json
import socket
import ssl
import threading
import urllib.parse

_original_getaddrinfo = socket.getaddrinfo
_doh_cache = {}
_doh_lock = threading.Lock()
_doh_disabled = False

def _cloudflare_a_records(host):
    global _doh_disabled
    with _doh_lock:
        if _doh_disabled:
            return ()
        if host in _doh_cache:
            return _doh_cache[host]
        sock = None
        tls = None
        try:
            # Connect to Cloudflare by fixed IP while verifying TLS/SNI by hostname.
            sock = socket.create_connection(('1.1.1.1', 443), timeout=4)
            tls = ssl.create_default_context().wrap_socket(
                sock, server_hostname='cloudflare-dns.com'
            )
            query = urllib.parse.urlencode({'name': host, 'type': 'A'})
            request = (
                f'GET /dns-query?{query} HTTP/1.1\r\n'
                'Host: cloudflare-dns.com\r\n'
                'Accept: application/dns-json\r\n'
                'Connection: close\r\n\r\n'
            )
            tls.sendall(request.encode('ascii'))
            response = http.client.HTTPResponse(tls)
            response.begin()
            if response.status != 200:
                addresses = ()
            else:
                payload = json.loads(response.read())
                addresses = tuple(
                    answer['data'] for answer in payload.get('Answer', [])
                    if answer.get('type') == 1
                    and isinstance(answer.get('data'), str)
                    and not answer['data'].startswith('127.')
                )
        except Exception:
            addresses = ()
            _doh_disabled = True
        finally:
            if tls is not None:
                try:
                    tls.close()
                except Exception:
                    pass
            elif sock is not None:
                try:
                    sock.close()
                except Exception:
                    pass
        _doh_cache[host] = addresses
        return addresses

def _getaddrinfo_with_doh(host, port, family=0, type=0, proto=0, flags=0):
    try:
        return _original_getaddrinfo(host, port, family, type, proto, flags)
    except socket.gaierror as original_error:
        if not isinstance(host, str) or family not in (0, socket.AF_UNSPEC, socket.AF_INET):
            raise original_error
        try:
            ipaddress.ip_address(host)
            raise original_error
        except ValueError:
            pass
        addresses = _cloudflare_a_records(host)
        if not addresses:
            raise original_error
        port_value = int(port) if isinstance(port, str) and port.isdigit() else port
        socktype = type or socket.SOCK_STREAM
        protocol = proto or (socket.IPPROTO_TCP if socktype == socket.SOCK_STREAM else 0)
        return [
            (socket.AF_INET, socktype, protocol, '', (address, port_value))
            for address in addresses
        ]

socket.getaddrinfo = _getaddrinfo_with_doh
"""

def _pip_install(command):
    from collections import deque
    import os
    import time

    transient_network_markers = (
        'NameResolutionError',
        'Temporary failure in name resolution',
        'Failed to resolve',
        'Could not resolve host',
        'Connection timed out',
        'ReadTimeoutError',
        'NewConnectionError',
        'Network is unreachable',
    )
    dns_bootstrap_dir = WORKDIR / 'pip_dns_bootstrap'
    dns_bootstrap_dir.mkdir(parents=True, exist_ok=True)
    dns_bootstrap_file = dns_bootstrap_dir / 'sitecustomize.py'
    dns_bootstrap_file.write_text(_DNS_BOOTSTRAP_SOURCE.lstrip(), encoding='utf-8')
    import importlib.util
    dns_spec = importlib.util.spec_from_file_location('_qwen_dns_bootstrap', dns_bootstrap_file)
    dns_module = importlib.util.module_from_spec(dns_spec)
    dns_spec.loader.exec_module(dns_module)
    pip_env = os.environ.copy()
    existing_pythonpath = pip_env.get('PYTHONPATH')
    pip_env['PYTHONPATH'] = os.pathsep.join(
        part for part in (str(dns_bootstrap_dir), existing_pythonpath) if part
    )

    wheelhouse_manifests = list(Path('/kaggle/input').rglob('qwen_offline_wheelhouse.json'))
    if len(wheelhouse_manifests) > 1:
        raise RuntimeError('Tìm thấy nhiều offline wheelhouse; cần đúng một Kaggle Input.')
    if wheelhouse_manifests:
        import hashlib
        import shutil
        manifest_path = wheelhouse_manifests[0]
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        opaque_sdist_manifests = list(Path('/kaggle/input').rglob('qwen_opaque_sdists.json'))
        if len(opaque_sdist_manifests) != 1:
            raise RuntimeError('Cần đúng một Kaggle Input chứa các gói nguồn đã đổi tên.')
        opaque_manifest_path = opaque_sdist_manifests[0]
        opaque_manifest = json.loads(opaque_manifest_path.read_text(encoding='utf-8'))
        opaque_by_name = {
            item['filename']: item for item in opaque_manifest.get('artifacts', [])
        }
        opaque_files = {}
        for opaque_item in opaque_manifest.get('artifacts', []):
            opaque_matches = [
                path for path in opaque_manifest_path.parent.rglob(opaque_item['opaque_filename'])
                if path.is_file()
            ]
            if len(opaque_matches) != 1:
                raise RuntimeError('Không tìm thấy đúng một file gói nguồn đổi tên trong Kaggle Input.')
            opaque_path = opaque_matches[0]
            if opaque_path.stat().st_size != opaque_item['size']:
                raise RuntimeError('Kích thước gói nguồn đổi tên không khớp manifest.')
            opaque_digest = hashlib.sha256(opaque_path.read_bytes()).hexdigest()
            if opaque_digest != opaque_item['sha256']:
                raise RuntimeError('SHA-256 gói nguồn đổi tên không khớp manifest.')
            opaque_files[opaque_item['filename']] = opaque_path

        install_wheelhouse = WORKDIR / 'qwen_offline_wheelhouse'
        install_wheelhouse.mkdir(parents=True, exist_ok=True)
        invalid_artifacts = []
        for artifact in manifest.get('artifacts', []):
            filename = artifact['filename']
            if Path(filename).name != filename:
                invalid_artifacts.append(artifact['filename'])
                continue
            if filename.endswith(('.tar.gz', '.zip')):
                artifact_path = opaque_files.get(filename)
            else:
                artifact_matches = [
                    path for path in manifest_path.parent.rglob(filename)
                    if path.is_file()
                ]
                artifact_path = artifact_matches[0] if len(artifact_matches) == 1 else None
            if artifact_path is None or artifact_path.stat().st_size != artifact['size']:
                invalid_artifacts.append(filename)
                continue
            digest = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
            if digest != artifact['sha256']:
                invalid_artifacts.append(filename)
                continue
            shutil.copyfile(artifact_path, install_wheelhouse / filename)
        if set(opaque_by_name) != {
            artifact['filename'] for artifact in manifest.get('artifacts', [])
            if artifact['filename'].endswith(('.tar.gz', '.zip'))
        }:
            invalid_artifacts.append('opaque_source_manifest')
        if invalid_artifacts:
            raise RuntimeError(f'Offline wheelhouse thiếu/sai hash ở {len(invalid_artifacts)} artifact.')

        pip_command = list(command)
        install_pos = pip_command.index('install')
        pip_command[install_pos + 1:install_pos + 1] = [
            '--no-index', '--find-links', str(install_wheelhouse), '--retries', '0', '--timeout', '20',
        ]
        print(f'Cài dependency offline từ Kaggle Input ({len(manifest.get("artifacts", []))} artifact đã xác minh).', flush=True)
        process = subprocess.Popen(pip_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1, env=pip_env)
        tail = deque(maxlen=300)
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line)
            _qwen_health.installer_activity()
        return_code = process.wait()
        log_text = ''.join(tail)
        redactor = globals().get('_qwen_redact')
        if callable(redactor):
            log_text = redactor(log_text)
        (WORKDIR / 'qwen_install_last.log').write_text(log_text, encoding='utf-8')
        if return_code:
            raise RuntimeError(f'Offline pip install exited with code {return_code}; xem qwen_install_last.log.')
        return

    index_error_markers = transient_network_markers + (
        'No matching distribution found',
        'Could not find a version that satisfies the requirement',
    )
    indexes = (
        ('PyPI', None),
        ('Tsinghua mirror', 'https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple'),
        ('Huawei mirror', 'https://mirrors.huaweicloud.com/repository/pypi/simple'),
        ('SUSTech mirror', 'https://mirrors.sustech.edu.cn/pypi/web/simple'),
        ('Aliyun mirror', 'https://mirrors.aliyun.com/pypi/simple/'),
    )
    return_code = 1
    log_text = ''
    for index_name, index_url in indexes:
        pip_command = list(command)
        install_pos = pip_command.index('install')
        pip_options = ['--retries', '1', '--timeout', '25']
        if index_url:
            pip_options.extend(['--index-url', index_url])
        pip_command[install_pos + 1:install_pos + 1] = pip_options

        tail = deque(maxlen=300)
        process = subprocess.Popen(pip_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1, env=pip_env)
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line)
            _qwen_health.installer_activity()
        return_code = process.wait()
        log_text = ''.join(tail)
        if not return_code:
            break
        if not any(marker in log_text for marker in index_error_markers):
            break
        if (index_name, index_url) != indexes[-1]:
            print(f'pip không lấy được package từ {index_name}; chuyển sang nguồn dự phòng.', flush=True)

    redactor = globals().get('_qwen_redact')
    if callable(redactor):
        log_text = redactor(log_text)
    (WORKDIR / 'qwen_install_last.log').write_text(log_text, encoding='utf-8')
    if return_code:
        raise RuntimeError(f'pip install exited with code {return_code}; last output follows:\n{log_text[-10000:]}')

if PACKAGE_LOCK.exists():
    locked_packages = json.loads(PACKAGE_LOCK.read_text(encoding='utf-8'))
    if locked_packages.get('torch'):
        locked_packages['torch'] = Version(locked_packages['torch']).base_version
    locked_transformers = Version(locked_packages['transformers'])
    if locked_transformers < MIN_SUPPORTED_TRANSFORMERS:
        print(
            f'Lock Transformers cũ ({locked_transformers}); nâng lên {FAST_TRANSFORMERS_VERSION} '
            'để dùng phiên bản Transformers đã khóa cho Qwen3 AWQ.'
        )
        locked_packages['transformers'] = FAST_TRANSFORMERS_VERSION
    current_packages = installed_package_versions()
    if current_packages != locked_packages:
        exact_specs = [f'{name}=={locked_packages[name]}' for name in PACKAGE_NAMES]
        torch_constraint = Path('/tmp/qwen3_14b_awq_torch_constraint.txt')
        torch_constraint.write_text(f"torch=={importlib.metadata.version('torch')}\n", encoding='utf-8')
        _pip_install([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--constraint', str(torch_constraint), *exact_specs])
        importlib.invalidate_caches()
        current_packages = installed_package_versions()
        if current_packages != locked_packages:
            raise RuntimeError(f'Không khôi phục được môi trường đã khóa. Mong đợi {locked_packages}; hiện có {current_packages}.')
        tmp_lock = PACKAGE_LOCK.with_suffix('.json.tmp')
        tmp_lock.write_text(json.dumps(locked_packages, indent=2), encoding='utf-8')
        tmp_lock.replace(PACKAGE_LOCK)
else:
    torch_constraint = Path('/tmp/qwen3_14b_awq_torch_constraint.txt')
    torch_constraint.write_text(f"torch=={importlib.metadata.version('torch')}\n", encoding='utf-8')
    _pip_install([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--constraint', str(torch_constraint), *PACKAGE_SPECS])
    importlib.invalidate_caches()
    locked_packages = installed_package_versions()
    tmp_lock = PACKAGE_LOCK.with_suffix('.json.tmp')
    tmp_lock.write_text(json.dumps(locked_packages, indent=2), encoding='utf-8')
    tmp_lock.replace(PACKAGE_LOCK)

tmp_lock = PACKAGE_LOCK.with_suffix('.json.normalized.tmp')
tmp_lock.write_text(json.dumps(locked_packages, indent=2), encoding='utf-8')
tmp_lock.replace(PACKAGE_LOCK)
from transformers.utils.import_utils import is_gptqmodel_available
if not is_gptqmodel_available():
    raise RuntimeError('gptqmodel chưa được cài/nhận diện; xem lỗi cài package ở cell này rồi chạy lại.')
print('gptqmodel:', importlib.metadata.version('gptqmodel'))
print('Môi trường khóa Kaggle:', json.dumps(locked_packages, indent=2))

_qwen_health.cell_finished(final=False)


## Dịch trong tiến trình Python mới
Sau khi cài dependency, worker mới thực hiện kiểm tra dữ liệu/GPU, smoke, full, QA và xuất checkpoint. Cách này tránh trộn NumPy đã nạp trong kernel với phiên bản vừa cài. Theo dõi `[qwen-health]` và `qwen_health.json`.


In [ ]:
# Pip changed NumPy on disk; execute translation in a fresh interpreter.
import subprocess as _worker_subprocess
import sys as _worker_sys
_qwen_health.close()
_qwen_health.worker.join(timeout=5)
_parent_ip = __import__('IPython').get_ipython()
if _parent_ip is not None:
    for _hook_name in ('_qwen_health_post_run', '_qwen_kaggle_error_hook'):
        _hook = getattr(_parent_ip, _hook_name, None)
        if _hook is not None:
            try:
                _parent_ip.events.unregister('post_run_cell', _hook)
            except Exception:
                pass
_worker_path = WORKDIR / 'qwen_fresh_worker.py'
_worker_path.write_text('import json as _worker_json\ntry:\n    exec(compile(\'"""Small, privacy-safe notebook heartbeat; safe to embed before notebook cells."""\\nimport contextlib as _health_contextlib\\nimport json as _health_json\\nimport os as _health_os\\nimport re as _health_re\\nimport subprocess as _health_subprocess\\nimport threading as _health_threading\\nimport time as _health_time\\nfrom datetime import datetime as _HealthDateTime, timezone as _HealthTimezone\\nfrom pathlib import Path as _HealthPath\\n\\n\\nclass QwenHealthMonitor:\\n    def __init__(self, path, interval=30, clock=None):\\n        self.path = _HealthPath(path)\\n        self.path.parent.mkdir(parents=True, exist_ok=True)\\n        self.clock = clock or _health_time.monotonic\\n        self.interval = interval\\n        self.lock = _health_threading.RLock()\\n        self.stop_event = _health_threading.Event()\\n        self.started = self.clock()\\n        self.last_progress = self.started\\n        self.last_emit = -float(\\\'inf\\\')\\n        self.last_signature = None\\n        self.payload = {\\\'phase\\\': \\\'initializing\\\', \\\'cell\\\': None, \\\'state\\\': \\\'starting\\\'}\\n        self.worker = _health_threading.Thread(target=self._run, daemon=True)\\n        self.worker.start()\\n\\n    def set_phase(self, phase, cell=None):\\n        if not isinstance(phase, str) or not _health_re.fullmatch(r\\\'[a-z_]{1,48}\\\', phase):\\n            phase = \\\'unknown\\\'\\n        with self.lock:\\n            self.payload.update(phase=phase, state=\\\'running\\\')\\n            if cell is not None:\\n                self.payload[\\\'cell\\\'] = int(cell)\\n            self.last_progress = self.clock()\\n            self.last_signature = None\\n            for key in (\\\'progress_done\\\', \\\'progress_total\\\', \\\'progress_unit\\\'):\\n                self.payload.pop(key, None)\\n        self.emit(force=True)\\n\\n    def progress(self, progress=None, **kwargs):\\n        values = dict(progress or {})\\n        values.update(kwargs)\\n        allowed = (\\\'processed_this_session\\\', \\\'completed_this_session\\\', \\\'errors_this_session\\\',\\n                   \\\'total_this_session\\\', \\\'completed_before_session\\\', \\\'progress_done\\\', \\\'progress_total\\\')\\n        with self.lock:\\n            phase = values.get(\\\'phase\\\')\\n            if isinstance(phase, str) and _health_re.fullmatch(r\\\'[a-z_]{1,48}\\\', phase):\\n                self.payload[\\\'phase\\\'] = phase\\n            for key in allowed:\\n                value = values.get(key)\\n                if isinstance(value, (int, float)) and not isinstance(value, bool):\\n                    self.payload[key] = max(0, int(value))\\n            unit = values.get(\\\'progress_unit\\\')\\n            if unit in (\\\'bytes\\\', \\\'installer_lines\\\', \\\'titles\\\', \\\'files\\\'):\\n                self.payload[\\\'progress_unit\\\'] = unit\\n            signature = (self.payload[\\\'phase\\\'],) + tuple(self.payload.get(k) for k in allowed)\\n            if signature != self.last_signature:\\n                self.last_progress = self.clock()\\n                self.last_signature = signature\\n        self.emit()\\n\\n    def installer_activity(self):\\n        with self.lock:\\n            done = self.payload.get(\\\'progress_done\\\', 0) + 1\\n        self.progress(progress_done=done, progress_unit=\\\'installer_lines\\\')\\n\\n    def cell_finished(self, final=False):\\n        with self.lock:\\n            self.payload[\\\'state\\\'] = \\\'completed\\\' if final else \\\'waiting_next_cell\\\'\\n            self.last_progress = self.clock()\\n        self.emit(force=True)\\n        if final:\\n            self.stop_event.set()\\n\\n    def fail(self, error):\\n        with self.lock:\\n            self.payload.update(state=\\\'error\\\', last_error_class=type(error).__name__)\\n        self.emit(force=True)\\n        self.stop_event.set()\\n\\n    def snapshot(self):\\n        with self.lock:\\n            now = self.clock()\\n            age = max(0, now - self.last_progress)\\n            payload = dict(self.payload)\\n            terminal = payload[\\\'state\\\'] in (\\\'completed\\\', \\\'error\\\')\\n            threshold = {\\\'install_dependencies\\\': 900, \\\'verify_model\\\': 600,\\n                         \\\'loading_model\\\': 600, \\\'generation\\\': 300}.get(payload[\\\'phase\\\'], 600)\\n            if not terminal and payload[\\\'state\\\'] != \\\'waiting_next_cell\\\':\\n                payload[\\\'state\\\'] = \\\'suspected_stall\\\' if age > threshold else \\\'running\\\'\\n            payload.update(heartbeat_at_utc=_HealthDateTime.now(_HealthTimezone.utc).isoformat(),\\n                           last_progress_at_utc=_HealthDateTime.fromtimestamp(\\n                               _health_time.time() - age, _HealthTimezone.utc).isoformat(),\\n                           progress_age_seconds=round(age, 1), elapsed_seconds=round(now-self.started, 1),\\n                           stall_threshold_seconds=threshold, process_id=_health_os.getpid(),\\n                           process_cpu_seconds=round(sum(_health_os.times()[:2]), 2))\\n            checkpoint = self.path.parent / \\\'cache_qwen3_14b_awq.sqlite3\\\'\\n            try:\\n                stat = checkpoint.stat()\\n                payload.update(checkpoint_bytes=stat.st_size,\\n                               checkpoint_age_seconds=round(max(0, _health_time.time()-stat.st_mtime), 1))\\n            except OSError:\\n                payload[\\\'checkpoint_available\\\'] = False\\n            return payload\\n\\n    def emit(self, force=False, gpu=False):\\n        with self.lock:\\n            if not force and self.clock()-self.last_emit < 10:\\n                return\\n            payload = self.snapshot()\\n            if gpu:\\n                try:\\n                    result = _health_subprocess.run(\\n                        [\\\'nvidia-smi\\\', \\\'--query-gpu=utilization.gpu,memory.used\\\', \\\'--format=csv,noheader,nounits\\\'],\\n                        capture_output=True, text=True, timeout=3)\\n                    row = result.stdout.splitlines()[0].split(\\\',\\\')\\n                    payload.update(gpu_utilization_percent=int(row[0].strip()), gpu_memory_used_mib=int(row[1].strip()))\\n                except Exception:\\n                    payload[\\\'gpu_metrics_available\\\'] = False\\n            temporary = self.path.with_suffix(\\\'.json.tmp\\\')\\n            try:\\n                temporary.write_text(_health_json.dumps(payload, ensure_ascii=True), encoding=\\\'utf-8\\\')\\n                temporary.replace(self.path)\\n            except Exception as exc:\\n                payload[\\\'health_write_error_class\\\'] = type(exc).__name__\\n            print(\\\'[qwen-health] \\\' + _health_json.dumps(payload, ensure_ascii=True), flush=True)\\n            self.last_emit = self.clock()\\n\\n    @_health_contextlib.contextmanager\\n    def stage(self, phase, cell=None, final=False):\\n        self.set_phase(phase, cell)\\n        try:\\n            yield\\n        except BaseException as exc:\\n            with self.lock:\\n                self.payload.update(state=\\\'error\\\', last_error_class=type(exc).__name__)\\n            self.emit(force=True)\\n            self.stop_event.set()\\n            raise\\n        else:\\n            with self.lock:\\n                self.payload[\\\'state\\\'] = \\\'completed\\\' if final else \\\'waiting_next_cell\\\'\\n                self.last_progress = self.clock()\\n            self.emit(force=True)\\n            if final:\\n                self.stop_event.set()\\n\\n    def _run(self):\\n        while not self.stop_event.wait(self.interval):\\n            self.emit(force=True, gpu=True)\\n\\n    def close(self):\\n        self.stop_event.set()\\n\\n_qwen_health = QwenHealthMonitor(_HealthPath(\\\'/kaggle/working/title_translation_cado_qwen3_14b_awq_full_v3_sampled/qwen_health.json\\\'))\\ndef _qwen_health_post_run(result):\\n    error = getattr(result, \\\'error_before_exec\\\', None) or getattr(result, \\\'error_in_exec\\\', None)\\n    if error is not None:\\n        _qwen_health.fail(error)\\n_health_ip = __import__(\\\'IPython\\\').get_ipython()\\nif _health_ip is not None:\\n    _previous_health_hook = getattr(_health_ip, \\\'_qwen_health_post_run\\\', None)\\n    if _previous_health_hook is not None:\\n        try:\\n            _health_ip.events.unregister(\\\'post_run_cell\\\', _previous_health_hook)\\n        except Exception:\\n            pass\\n    _health_ip.events.register(\\\'post_run_cell\\\', _qwen_health_post_run)\\n    _health_ip._qwen_health_post_run = _qwen_health_post_run\\n\\n_qwen_health.set_phase(\\\'prepare_inputs\\\', cell=2)\\n# Kaggle has read-only inputs and writable outputs; never mount or change Google Drive here.\\nimport json\\nimport re\\nimport shutil\\nimport zipfile\\nfrom pathlib import Path, PurePosixPath\\nfrom datetime import datetime, timezone\\nfrom IPython import get_ipython\\n\\n# Check the actual runtime before dependency installation or model-file hashing.\\n_qwen_health.set_phase(\\\'preflight_gpu\\\', cell=2)\\nimport torch\\n_gpu_probe = {\\n    \\\'torch\\\': torch.__version__,\\n    \\\'cuda_available\\\': torch.cuda.is_available(),\\n    \\\'nvidia_device_present\\\': Path(\\\'/dev/nvidia0\\\').exists(),\\n    \\\'nvidia_smi_available\\\': bool(shutil.which(\\\'nvidia-smi\\\')),\\n}\\nprint(\\\'[qwen-boot] \\\' + json.dumps(_gpu_probe), flush=True)\\nif not torch.cuda.is_available():\\n    raise RuntimeError(\\\'Kaggle chưa cấp CUDA GPU. Vào Edit notebook, Settings / Session options, chọn GPU T4 x2 rồi bắt đầu lại session; không cài dependency trên runtime CPU.\\\')\\n_gpu_properties = torch.cuda.get_device_properties(0)\\nif _gpu_properties.total_memory < 14 * 1024**3 or (_gpu_properties.major, _gpu_properties.minor) < (7, 5):\\n    raise RuntimeError(\\\'GPU hiện tại không đủ VRAM hoặc compute capability để chạy Qwen3-14B-AWQ ExLlamaV2.\\\')\\n\\nKAGGLE_INPUT_ROOT = Path(\\\'/kaggle/input\\\')\\nKAGGLE_WORK_ROOT = Path(\\\'/kaggle/working\\\')\\nWORKDIR = KAGGLE_WORK_ROOT / \\\'title_translation_cado_qwen3_14b_awq_full_v3_sampled\\\'\\nWORKDIR.mkdir(parents=True, exist_ok=True)\\nLOCAL_DIR = Path(\\\'/kaggle/temp/title_translation_cado_qwen3_14b_awq_full_v3_sampled\\\')\\nLOCAL_DIR.mkdir(parents=True, exist_ok=True)\\nLOCAL_CACHE = LOCAL_DIR / \\\'cache_qwen3_14b_awq.sqlite3\\\'\\nWORKING_CACHE = WORKDIR / \\\'cache_qwen3_14b_awq.sqlite3\\\'\\nBUNDLE_NAME = \\\'title_translation_qwen3_14b_awq_colab_v6_sampled_autoreload_bundle.zip\\\'\\nLEGACY_RUN_DIR_NAME = \\\'title_translation_cado_qwen3_14b_awq_full_v2\\\'\\nCHECKPOINT_ZIP_NAME = \\\'qwen3_14b_awq_kaggle_checkpoint.zip\\\'\\n\\nif not KAGGLE_INPUT_ROOT.exists():\\n    raise RuntimeError(\\\'Không thấy /kaggle/input. Hãy mở notebook trong Kaggle và đính kèm Kaggle Dataset đầu vào.\\\')\\n\\nbundle_candidates = [\\n    path for path in KAGGLE_INPUT_ROOT.rglob(BUNDLE_NAME)\\n    if path.is_file()\\n]\\nif len(bundle_candidates) == 1:\\n    BUNDLE_INPUT = bundle_candidates[0]\\nelif len(bundle_candidates) == 0:\\n    # Kaggle may recursively unpack both the outer Dataset ZIP and this\\n    # nested bundle ZIP. In that case the bundle is an extracted directory,\\n    # not a file with BUNDLE_NAME. Repack only its two known source files in\\n    # the same flat layout expected by the rest of this setup cell.\\n    bundle_dir_names = {BUNDLE_NAME, BUNDLE_NAME[:-4]}\\n    unpacked_bundle_dirs = [\\n        path for path in KAGGLE_INPUT_ROOT.rglob(\\\'*\\\')\\n        if path.is_dir() and path.name in bundle_dir_names\\n    ]\\n    if len(unpacked_bundle_dirs) != 1:\\n        raise FileNotFoundError(\\n            \\\'Cần đúng 1 bundle v6 trong Kaggle Input; \\\'\\n            f\\\'tìm thấy {len(bundle_candidates)} file ZIP và \\\'\\n            f\\\'{len(unpacked_bundle_dirs)} thư mục bundle đã giải nén. \\\'\\n            \\\'Hãy đính kèm private dataset theo guide.\\\'\\n        )\\n    expected_bundle_files = (\\n        \\\'title_translation_hybrid.py\\\',\\n        \\\'title_translation_pipeline.py\\\',\\n    )\\n    unpacked_bundle_dir = unpacked_bundle_dirs[0]\\n    unpacked_files = {}\\n    for name in expected_bundle_files:\\n        matches = [path for path in unpacked_bundle_dir.rglob(name) if path.is_file()]\\n        if len(matches) != 1:\\n            raise FileNotFoundError(\\n                f\\\'Thư mục bundle đã bung phải có đúng một {name}; \\\'\\n                f\\\'tìm thấy {len(matches)}.\\\'\\n            )\\n        unpacked_files[name] = matches[0]\\n    repacked_bundle = LOCAL_DIR / BUNDLE_NAME\\n    with zipfile.ZipFile(repacked_bundle, \\\'w\\\', compression=zipfile.ZIP_DEFLATED) as archive:\\n        for name, path in unpacked_files.items():\\n            archive.write(path, arcname=name)\\n    BUNDLE_INPUT = repacked_bundle\\n    print(\\\'Kaggle đã tự bung bundle; đã đóng gói lại hai file nguồn:\\\', unpacked_bundle_dir)\\nelse:\\n    raise FileNotFoundError(\\n        f\\\'Cần đúng 1 bundle v6 trong Kaggle Input; hiện tìm thấy {len(bundle_candidates)} file ZIP. \\\'\\n        \\\'Hãy chỉ đính kèm một private dataset theo guide.\\\'\\n    )\\nLEGACY_CACHE_NAME = \\\'cache_qwen3_14b_awq.sqlite3\\\'\\nlegacy_cache_candidates = [\\n    path for path in KAGGLE_INPUT_ROOT.rglob(LEGACY_CACHE_NAME)\\n    if path.parent.name == LEGACY_RUN_DIR_NAME\\n    and (path.parent / \\\'hybrid_environment_lock.json\\\').is_file()\\n]\\nif len(legacy_cache_candidates) > 1:\\n    raise RuntimeError(\\\'Tìm thấy nhiều checkpoint v2; hãy chỉ đính kèm một bản checkpoint nguồn.\\\')\\nLEGACY_WORKDIR = legacy_cache_candidates[0].parent if legacy_cache_candidates else None\\n\\n# A checkpoint ZIP exported by this notebook can restore a v6 run when Kaggle\\n# Files persistence is unavailable. Only flat, known runtime files are allowed.\\nallowed_snapshot_files = {\\n    \\\'cache_qwen3_14b_awq.sqlite3\\\', \\\'hybrid_environment_lock.json\\\',\\n    \\\'hybrid_run_state.json\\\', \\\'legacy_cache_import.json\\\',\\n    \\\'qwen3_14b_awq_kaggle_package_lock.json\\\', \\\'qwen_live_status.json\\\', \\\'qwen_health.json\\\',\\n}\\nsnapshot_candidates = list(KAGGLE_INPUT_ROOT.rglob(CHECKPOINT_ZIP_NAME))\\nif len(snapshot_candidates) > 1:\\n    raise RuntimeError(\\\'Tìm thấy nhiều ZIP checkpoint Kaggle; chỉ đính kèm bản mới nhất.\\\')\\nif snapshot_candidates:\\n    with zipfile.ZipFile(snapshot_candidates[0]) as archive:\\n        for member in archive.infolist():\\n            path = PurePosixPath(member.filename)\\n            if path.is_absolute() or len(path.parts) != 1 or \\\'..\\\' in path.parts:\\n                raise RuntimeError(\\\'ZIP checkpoint có đường dẫn không an toàn; không giải nén.\\\')\\n            if path.name not in allowed_snapshot_files:\\n                raise RuntimeError(f\\\'Tệp không được phép trong ZIP checkpoint: {path.name}\\\')\\n            destination = WORKDIR / path.name\\n            if not destination.exists() and not member.is_dir():\\n                destination.write_bytes(archive.read(member))\\n    print(\\\'Đã khôi phục snapshot checkpoint Kaggle vào:\\\', WORKDIR)\\n\\nBUNDLE_PATH = WORKDIR / BUNDLE_NAME\\nif BUNDLE_PATH.exists():\\n    import hashlib\\n    old_sha = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()\\n    input_sha = hashlib.sha256(BUNDLE_INPUT.read_bytes()).hexdigest()\\n    has_prior_run = WORKING_CACHE.exists() or (WORKDIR / \\\'hybrid_run_state.json\\\').exists()\\n    if old_sha != input_sha and has_prior_run:\\n        raise RuntimeError(\\n            \\\'Bundle Kaggle khác với bundle đã dùng để tạo checkpoint. \\\'\\n            \\\'Không tự thay code khi resume; hãy dừng và yêu cầu kiểm tra trước.\\\'\\n        )\\n    if old_sha != input_sha:\\n        shutil.copy2(BUNDLE_INPUT, BUNDLE_PATH)\\nelse:\\n    shutil.copy2(BUNDLE_INPUT, BUNDLE_PATH)\\n\\nif WORKING_CACHE.exists() and not LOCAL_CACHE.exists():\\n    shutil.copy2(WORKING_CACHE, LOCAL_CACHE)\\n    print(\\\'Đã khôi phục cache v6 từ /kaggle/working.\\\')\\n\\nhas_v6_checkpoint = WORKING_CACHE.exists() or (WORKDIR / \\\'legacy_cache_import.json\\\').exists()\\nif not has_v6_checkpoint and LEGACY_WORKDIR is None:\\n    raise FileNotFoundError(\\n        \\\'Thiếu cache nguồn v2. Đính kèm private Kaggle Dataset có \\\'\\n        \\\'title_translation_cado_qwen3_14b_awq_full_v2/cache_qwen3_14b_awq.sqlite3 \\\'\\n        \\\'và hybrid_environment_lock.json; xem guide đi kèm.\\\'\\n    )\\n\\nif LEGACY_WORKDIR is not None:\\n    print(\\\'Đã tìm thấy checkpoint v2 để nhập chỉ-đọc:\\\', LEGACY_WORKDIR)\\nelse:\\n    print(\\\'Tiếp tục checkpoint v6 đã có; không cần nhập lại cache v2.\\\')\\nprint(\\\'Kaggle run folder:\\\', WORKDIR)\\nprint(\\\'Cache local runtime:\\\', LOCAL_CACHE)\\n\\n# Write privacy-safe status locally; there is no Google Drive mount or remote\\n# code-reload channel in the Kaggle port.\\nimport traceback\\n_qwen_status_path = WORKDIR / \\\'qwen_live_status.json\\\'\\ndef _qwen_redact(value):\\n    value = str(value)\\n    value = re.sub(r\\\'hf_[A-Za-z0-9]{16,}\\\', \\\'[REDACTED_HF_TOKEN]\\\', value)\\n    value = re.sub(r\\\'AIza[0-9A-Za-z_-]{20,}\\\', \\\'[REDACTED_GOOGLE_KEY]\\\', value)\\n    value = re.sub(r\\\'AQ\\\\.[A-Za-z0-9_-]{20,}\\\', \\\'[REDACTED_API_KEY]\\\', value)\\n    return value\\ndef _qwen_kaggle_error_hook(result):\\n    error = getattr(result, \\\'error_before_exec\\\', None) or getattr(result, \\\'error_in_exec\\\', None)\\n    if error is None:\\n        raw_cell = getattr(getattr(result, \\\'info\\\', None), \\\'raw_cell\\\', \\\'\\\') or \\\'\\\'\\n        if \\\'_qwen_full_run_cell_marker\\\' not in raw_cell:\\n            return\\n        summary = globals().get(\\\'full_result\\\')\\n        if not isinstance(summary, dict):\\n            return\\n        try:\\n            payload = {\\n                \\\'state\\\': summary.get(\\\'status\\\', \\\'unknown\\\'),\\n                \\\'phase\\\': \\\'run_cell_finished\\\',\\n                \\\'updated_at_utc\\\': datetime.now(timezone.utc).isoformat(),\\n                \\\'completed_unique_titles\\\': summary.get(\\\'completed_unique_titles\\\'),\\n                \\\'total_unique_titles\\\': summary.get(\\\'total_unique_titles\\\'),\\n                \\\'remaining_unique_titles\\\': summary.get(\\\'remaining_unique_titles\\\'),\\n                \\\'generation\\\': summary.get(\\\'generation\\\'),\\n            }\\n            temp = _qwen_status_path.with_suffix(\\\'.json.tmp\\\')\\n            temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\n            temp.replace(_qwen_status_path)\\n        except Exception as log_error:\\n            print(f\\\'[qwen-kaggle-observer] status write failed: {type(log_error).__name__}\\\', flush=True)\\n        return\\n    try:\\n        payload = {\\n            \\\'state\\\': \\\'error\\\',\\n            \\\'phase\\\': \\\'notebook_cell_error\\\',\\n            \\\'updated_at_utc\\\': datetime.now(timezone.utc).isoformat(),\\n            \\\'cell_execution_count\\\': getattr(result, \\\'execution_count\\\', None),\\n            \\\'last_error_class\\\': type(error).__name__,\\n        }\\n        temp = _qwen_status_path.with_suffix(\\\'.json.tmp\\\')\\n        temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\n        temp.replace(_qwen_status_path)\\n    except Exception as log_error:\\n        print(f\\\'[qwen-kaggle-observer] status write failed: {type(log_error).__name__}\\\', flush=True)\\n_qwen_ip = get_ipython()\\nif _qwen_ip is not None:\\n    _old_hook = getattr(_qwen_ip, \\\'_qwen_kaggle_error_hook\\\', None)\\n    if _old_hook is not None:\\n        try: _qwen_ip.events.unregister(\\\'post_run_cell\\\', _old_hook)\\n        except Exception: pass\\n    _qwen_ip.events.register(\\\'post_run_cell\\\', _qwen_kaggle_error_hook)\\n    _qwen_ip._qwen_kaggle_error_hook = _qwen_kaggle_error_hook\\n\\n_qwen_health.cell_finished(final=False)\\n\\n\\n\\n_qwen_health.set_phase(\\\'verify_source\\\', cell=4)\\n# Nạp mã pipeline hiện tại từ bundle và tải đúng Parquet nguồn công khai.\\nimport hashlib\\nimport importlib\\nimport io\\nimport json\\nimport sys\\nimport zipfile\\n\\n# Bundle đã được copy từ Kaggle Input vào WORKDIR ở cell chuẩn bị.\\nif not BUNDLE_PATH.is_file():\\n    raise FileNotFoundError(\\\'Thiếu bundle v6. Hãy đính kèm private Kaggle Dataset như mô tả ở đầu notebook.\\\')\\n\\nPROJECT_DIR = Path(\\\'/content/title_translation_qwen3_14b_awq_v6_sampled_source\\\')\\nPROJECT_DIR.mkdir(parents=True, exist_ok=True)\\nMAX_BUNDLE_BYTES = 10 * 1024 * 1024\\nEXPECTED_BUNDLE_FILES = {\\\'title_translation_hybrid.py\\\', \\\'title_translation_pipeline.py\\\'}\\n\\ndef load_pipeline_bundle(expected_sha256=None):\\n    bundle_bytes = BUNDLE_PATH.read_bytes()\\n    if len(bundle_bytes) > MAX_BUNDLE_BYTES:\\n        raise RuntimeError(f\\\'Bundle lớn bất thường: {len(bundle_bytes)} bytes.\\\')\\n    bundle_sha256 = hashlib.sha256(bundle_bytes).hexdigest()\\n    if expected_sha256 and bundle_sha256.lower() != str(expected_sha256).lower():\\n        raise RuntimeError(f\\\'SHA-256 bundle không khớp lệnh reload: {bundle_sha256}\\\')\\n    with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as zf:\\n        names = set(zf.namelist())\\n        if names != EXPECTED_BUNDLE_FILES:\\n            raise RuntimeError(f\\\'Bundle có tên tệp không được phép: {sorted(names)}\\\')\\n        for name in EXPECTED_BUNDLE_FILES:\\n            info = zf.getinfo(name)\\n            if info.file_size > 2 * 1024 * 1024:\\n                raise RuntimeError(f\\\'Module trong bundle lớn bất thường: {name}\\\')\\n            (PROJECT_DIR / name).write_bytes(zf.read(name))\\n    sys.path.insert(0, str(PROJECT_DIR)) if str(PROJECT_DIR) not in sys.path else None\\n    importlib.invalidate_caches()\\n    sys.modules.pop(\\\'title_translation_hybrid\\\', None)\\n    sys.modules.pop(\\\'title_translation_pipeline\\\', None)\\n    module = importlib.import_module(\\\'title_translation_hybrid\\\')\\n    return module, bundle_sha256\\n\\npipeline, ACTIVE_BUNDLE_SHA = load_pipeline_bundle()\\nprint(\\\'Đã nạp pipeline từ Kaggle Input; SHA-256:\\\', ACTIVE_BUNDLE_SHA)\\n\\nsource_name = Path(pipeline.DATASET_FILE).name\\nsource_candidates = [\\n    path for path in KAGGLE_INPUT_ROOT.rglob(source_name)\\n    if path.is_file()\\n]\\nif len(source_candidates) != 1:\\n    raise FileNotFoundError(\\n        f\\\'Cần đúng một Parquet nguồn trong Kaggle Input; tìm thấy {len(source_candidates)}.\\\'\\n    )\\nSOURCE = source_candidates[0]\\nactual_sha = pipeline.quality.sha256_file(SOURCE)\\nif actual_sha.lower() != pipeline.DATASET_SHA256.lower():\\n    raise ValueError(\\\'SHA-256 Parquet nguồn không khớp; dừng trước khi dịch.\\\')\\nprint(f\\\'Nguồn đã xác minh: {pipeline.EXPECTED_ROWS:,} dòng; {pipeline.EXPECTED_UNIQUE_TITLES:,} title duy nhất; SHA-256={actual_sha}\\\')\\n\\n_qwen_health.cell_finished(final=False)\\n\\n\\n\\n_qwen_health.set_phase(\\\'check_gpu\\\', cell=5)\\n# Fail early unless Kaggle actually attached a compatible CUDA GPU.\\nimport torch\\nimport shutil\\nfrom pathlib import Path\\n\\nif not torch.cuda.is_available():\\n    raise RuntimeError(\\\'Chưa có CUDA GPU. Trong Kaggle Settings, chọn Accelerator = GPU rồi Start Session.\\\')\\ngpu = torch.cuda.get_device_properties(0)\\ngpu_vram_gib = gpu.total_memory / (1024 ** 3)\\ncompute_capability = (gpu.major, gpu.minor)\\nprint(f\\\'GPU: {gpu.name}; VRAM: {gpu_vram_gib:.1f} GiB; compute capability: {gpu.major}.{gpu.minor}\\\')\\nif compute_capability < (7, 5):\\n    raise RuntimeError(\\n        \\\'GPU này không đạt Turing (SM 7.5) trở lên cho ExLlamaV2 AWQ của v6. \\\'\\n        \\\'P100 không tương thích với kernel đã chọn; không bắt đầu dịch.\\\'\\n    )\\nif gpu.total_memory < 14 * 1024 ** 3:\\n    raise RuntimeError(\\\'Qwen3-14B AWQ cần tối thiểu 14 GiB VRAM trên GPU đầu tiên; không bắt đầu dịch.\\\')\\n\\ndef persist_qwen_cache(source_path):\\n    source_path = Path(source_path)\\n    if not source_path.exists():\\n        return\\n    temporary = WORKING_CACHE.with_name(WORKING_CACHE.name + \\\'.uploading\\\')\\n    shutil.copy2(source_path, temporary)\\n    temporary.replace(WORKING_CACHE)\\n\\nprint(\\\'Cache sẽ được đồng bộ định kỳ vào /kaggle/working; checkpoint:\\\', WORKING_CACHE)\\n\\n_qwen_health.cell_finished(final=False)\\n\\n\\n\\n_qwen_health.set_phase(\\\'prepare_smoke\\\', cell=7)\\nimport time\\nfrom datetime import datetime, timezone\\n# Preflight 20 title, sau đó chạy full ngay cả khi QA preflight báo cờ.\\n_qwen_full_run_cell_marker = \\\'qwen-full-run-kaggle-v6\\\'\\nQWEN_BATCH_SIZE = 1  # Giữ batch đã khóa khi smoke/checkpoint được tạo\\nQWEN_FULL_BATCH_SIZE = 4  # Thử tối đa; pipeline tự hạ 4 -> 2 -> 1 nếu OOM lúc generate\\nCHECKPOINT_EVERY_TITLES = 100\\nCHECKPOINT_EVERY_TITLES = 100\\nLIVE_STATUS_FILE = WORKDIR / \\\'qwen_live_status.json\\\'\\nCONTROL_POLL_SECONDS = 5\\n_control_last_status_write = 0.0\\n_SAFE_CONTROL_ERROR_REASONS = {\\\'repeated_phrase_guard\\\', \\\'truncated_output\\\', \\\'empty_output\\\'}\\n_SAFE_EXCEPTION_CLASSES = {\\\'RuntimeError\\\', \\\'ImportError\\\', \\\'ModuleNotFoundError\\\', \\\'CalledProcessError\\\', \\\'FileNotFoundError\\\', \\\'TimeoutError\\\', \\\'ConnectionError\\\', \\\'HTTPError\\\', \\\'ValueError\\\', \\\'TypeError\\\', \\\'KeyError\\\', \\\'AttributeError\\\', \\\'MemoryError\\\', \\\'OSError\\\'}\\n\\ndef _qwen_safe_recent_errors(items):\\n    safe_items = []\\n    for item in items if isinstance(items, list) else []:\\n        if not isinstance(item, dict):\\n            safe_items.append({\\\'reason\\\': \\\'other_error\\\'})\\n            continue\\n        reason = str(item.get(\\\'reason\\\') or \\\'\\\')\\n        safe_items.append({\\n            \\\'reason\\\': reason if reason in _SAFE_CONTROL_ERROR_REASONS else \\\'other_error\\\',\\n            \\\'truncated\\\': bool(item.get(\\\'truncated\\\')),\\n        })\\n    return safe_items[-3:]\\n\\ndef _qwen_safe_error_class(value):\\n    candidate = str(value or \\\'\\\').split(\\\':\\\', 1)[0].strip()\\n    return candidate if candidate in _SAFE_EXCEPTION_CLASSES else (\\\'unknown_error\\\' if value else None)\\n\\ndef qwen_kaggle_progress(progress):\\n    global _control_last_status_write\\n    _qwen_health.progress(progress)\\n    now = time.time()\\n    processed = int(progress.get(\\\'processed_this_session\\\', 0))\\n    if processed % 20 == 0 or now - _control_last_status_write >= 30 or bool(progress.get(\\\'last_error\\\')):\\n        status = {\\n            \\\'updated_at_utc\\\': datetime.now(timezone.utc).isoformat(),\\n            \\\'model_key\\\': progress.get(\\\'model_key\\\'),\\n            \\\'phase\\\': progress.get(\\\'phase\\\', \\\'unknown\\\'),\\n            \\\'action\\\': \\\'run\\\',\\n            \\\'processed_this_session\\\': processed,\\n            \\\'total_this_session\\\': int(progress.get(\\\'total_this_session\\\', 0)),\\n            \\\'completed_this_session\\\': int(progress.get(\\\'completed_this_session\\\', 0)),\\n            \\\'errors_this_session\\\': int(progress.get(\\\'errors_this_session\\\', 0)),\\n            \\\'completed_before_session\\\': int(progress.get(\\\'completed_before_session\\\', 0)),\\n            \\\'completed_estimate\\\': int(progress.get(\\\'completed_before_session\\\', 0)) + int(progress.get(\\\'completed_this_session\\\', 0)),\\n            \\\'elapsed_seconds\\\': float(progress.get(\\\'elapsed_seconds\\\', 0) or 0),\\n            \\\'throughput_titles_per_hour\\\': float(progress.get(\\\'throughput_titles_per_hour\\\', 0) or 0),\\n            \\\'recent_errors\\\': _qwen_safe_recent_errors(progress.get(\\\'recent_errors\\\', [])),\\n            \\\'last_error_class\\\': _qwen_safe_error_class(progress.get(\\\'last_error\\\')),\\n            \\\'checkpoint_file\\\': WORKING_CACHE.name,\\n            \\\'runtime\\\': \\\'Kaggle GPU; files persisted under /kaggle/working\\\',\\n            \\\'active_bundle_sha256\\\': ACTIVE_BUNDLE_SHA,\\n        }\\n        try:\\n            temp = LIVE_STATUS_FILE.with_suffix(\\\'.json.tmp\\\')\\n            temp.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\n            temp.replace(LIVE_STATUS_FILE)\\n            _control_last_status_write = now\\n        except Exception as exc:\\n            print(f\\\'[qwen-kaggle-status] write failed: {type(exc).__name__}\\\', flush=True)\\n    return \\\'run\\\'\\n\\ndef attach_durable_cache(context, enable_control=False):\\n    context[\\\'cache_paths\\\'] = {\\\'qwen3_14b_awq\\\': LOCAL_CACHE}\\n    context[\\\'cache_checkpoint_interval\\\'] = CHECKPOINT_EVERY_TITLES\\n    context[\\\'cache_checkpoint_callback\\\'] = persist_qwen_cache\\n    context[\\\'control_callback\\\'] = qwen_kaggle_progress\\n    context[\\\'control_poll_seconds\\\'] = CONTROL_POLL_SECONDS\\n    return context\\n\\ndef _migrate_legacy_qwen_cache(context):\\n    """Copy only reusable completed rows; never mutate the v2 source checkpoint."""\\n    import sqlite3\\n    from datetime import datetime, timezone\\n\\n    marker = WORKDIR / \\\'legacy_cache_import.json\\\'\\n    if marker.exists():\\n        print(\\\'Đã hoàn tất nhập cache v2 trước đó; giữ nguyên cache v3.\\\')\\n        return\\n\\n    legacy_workdir = LEGACY_WORKDIR\\n    if legacy_workdir is None:\\n        print(\\\'Không có cache v2 input; bỏ qua nhập cache cũ.\\\')\\n        marker.write_text(json.dumps({\\\'status\\\': \\\'no_legacy_cache\\\'}, indent=2), encoding=\\\'utf-8\\\')\\n        return\\n    legacy_cache = legacy_workdir / \\\'cache_qwen3_14b_awq.sqlite3\\\'\\n    legacy_lock_path = legacy_workdir / \\\'hybrid_environment_lock.json\\\'\\n    legacy_status_path = legacy_workdir / \\\'qwen_live_status.json\\\'\\n    if legacy_status_path.exists():\\n        try:\\n            legacy_status = json.loads(legacy_status_path.read_text(encoding=\\\'utf-8\\\'))\\n            last_state = str(legacy_status.get(\\\'state\\\', \\\'\\\')).lower()\\n            updated = datetime.fromisoformat(str(legacy_status.get(\\\'updated_at_utc\\\', \\\'\\\')).replace(\\\'Z\\\', \\\'+00:00\\\'))\\n            is_recent = (datetime.now(timezone.utc) - updated).total_seconds() < 1200\\n            if is_recent and last_state in {\\\'running\\\', \\\'generating\\\', \\\'loading_model\\\', \\\'awaiting_code_update\\\'}:\\n                raise RuntimeError(\\\'Run v2 có trạng thái hoạt động gần đây; hãy dừng hẳn run cũ trước khi nhập cache.\\\')\\n        except ValueError:\\n            pass\\n\\n    if not legacy_cache.exists() or not legacy_lock_path.exists():\\n        print(\\\'Không tìm thấy đủ cache/lock v2; v3 sẽ tiếp tục bằng cache riêng mới.\\\')\\n        marker.write_text(json.dumps({\\\'status\\\': \\\'no_legacy_cache\\\'}, indent=2), encoding=\\\'utf-8\\\')\\n        return\\n\\n    legacy_lock = json.loads(legacy_lock_path.read_text(encoding=\\\'utf-8\\\'))\\n    model_key = \\\'qwen3_14b_awq\\\'\\n    old_model = legacy_lock.get(\\\'models\\\', {}).get(model_key, {})\\n    old_revision = old_model.get(\\\'revision\\\')\\n    if old_model.get(\\\'model_id\\\') != pipeline.MODELS[model_key][\\\'model_id\\\']:\\n        raise RuntimeError(\\\'Cache v2 thuộc model khác; dừng để tránh trộn kết quả.\\\')\\n\\n    # Pin v3 to the immutable revision used by the old cache before computing\\n    # the new config ID. This allows safe reuse while recording the new decoder.\\n    if old_revision:\\n        new_lock = context[\\\'lock\\\']\\n        new_models = dict(new_lock[\\\'models\\\'])\\n        new_model = dict(new_models[model_key])\\n        new_model[\\\'revision\\\'] = old_revision\\n        new_models[model_key] = new_model\\n        new_config = pipeline._config_payload(\\n            new_models,\\n            qwen_batch_size=int(context.get(\\\'qwen_batch_size\\\', 1)),\\n        )\\n        new_config_id = pipeline.fingerprint(new_config)\\n        new_lock.update({\\\'models\\\': new_models, \\\'config\\\': new_config, \\\'config_id\\\': new_config_id})\\n        context[\\\'lock\\\'] = new_lock\\n        context[\\\'state\\\'][\\\'config_id\\\'] = new_config_id\\n        pipeline.atomic_json(WORKDIR / \\\'hybrid_environment_lock.json\\\', new_lock)\\n        pipeline.atomic_json(WORKDIR / \\\'hybrid_run_state.json\\\', context[\\\'state\\\'])\\n\\n    revision = context[\\\'lock\\\'][\\\'models\\\'][model_key][\\\'revision\\\']\\n    target_cache = pipeline.Cache(LOCAL_CACHE, model_key, revision, context[\\\'lock\\\'][\\\'config_id\\\'])\\n    imported = 0\\n    skipped = 0\\n    skipped_smoke = 0\\n    smoke_titles = set(context.get(\\\'samples\\\', {}).get(\\\'smoke\\\', []))\\n    with sqlite3.connect(f\\\'file:{legacy_cache.as_posix()}?mode=ro\\\', uri=True) as old_db:\\n        rows = old_db.execute(\\n            "SELECT source_title,output_text,status,truncated,input_tokens,output_tokens,attempts,"\\n            "api_requests,elapsed_seconds,error_text,model_version,response_id "\\n            "FROM translations"\\n        ).fetchall()\\n    for row in rows:\\n        title, output, status, truncated, in_tokens, out_tokens, attempts, api_requests, elapsed, error, version, response_id = row\\n        if str(title) in smoke_titles:\\n            # Force the smoke set through the new sampling decoder so the\\n            # preflight actually tests v6 instead of passing on legacy cache.\\n            skipped_smoke += 1\\n            continue\\n        record = {\\n            \\\'output_text\\\': output,\\n            \\\'status\\\': status,\\n            \\\'truncated\\\': bool(truncated),\\n            \\\'input_tokens\\\': in_tokens,\\n            \\\'output_tokens\\\': out_tokens,\\n            \\\'attempts\\\': attempts or 0,\\n            \\\'api_requests\\\': api_requests or attempts or 0,\\n            \\\'elapsed_seconds\\\': elapsed or 0.0,\\n            \\\'error_text\\\': error,\\n            \\\'model_version\\\': (version or \\\'legacy-qwen3-14b-awq\\\') + \\\'; legacy-greedy\\\',\\n            \\\'response_id\\\': response_id,\\n            \\\'generation_prompt_version\\\': \\\'legacy-qwen3-14b-awq-greedy-v1\\\',\\n        }\\n        if not pipeline._complete(record):\\n            skipped += 1\\n            continue\\n        target_cache.put(str(title), record)\\n        imported += 1\\n\\n    persist_qwen_cache(LOCAL_CACHE)\\n    payload = {\\n        \\\'status\\\': \\\'complete\\\',\\n        \\\'source_run_dir\\\': \\\'title_translation_cado_qwen3_14b_awq_full_v2\\\',\\n        \\\'source_cache_rows\\\': len(rows),\\n        \\\'imported_completed_nonrepetitive\\\': imported,\\n        \\\'skipped_incomplete_or_repetitive\\\': skipped,\\n        \\\'skipped_smoke_for_fresh_sampling_test\\\': skipped_smoke,\\n        \\\'source_revision\\\': old_revision,\\n        \\\'target_config_id\\\': context[\\\'lock\\\'][\\\'config_id\\\'],\\n        \\\'updated_at_utc\\\': datetime.now(timezone.utc).isoformat(),\\n    }\\n    temp_marker = marker.with_suffix(\\\'.json.tmp\\\')\\n    temp_marker.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\n    temp_marker.replace(marker)\\n    print(\\\'Nhập cache hoàn tất:\\\', json.dumps(payload, ensure_ascii=False))\\n\\n# Load the exact Qwen snapshot from a read-only Kaggle Input. The Hugging Face\\n# repository id and revision remain in the run lock; only model file transport\\n# is redirected to the verified local snapshot.\\nimport hashlib\\nimport os\\nimport json\\nfrom pathlib import Path\\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\\n\\nQWEN_MODEL_REPO = \\\'Qwen/Qwen3-14B-AWQ\\\'\\nQWEN_MODEL_REVISION = \\\'31c69efc29464b6bb0aee1398b5a7b50a99340c3\\\'\\nQWEN_MODEL_FILES = {\\n    \\\'.gitattributes\\\': {\\\'size\\\': 1570, \\\'git_sha1\\\': \\\'52373fe24473b1aa44333d318f578ae6bf04b49b\\\'},\\n    \\\'LICENSE\\\': {\\\'size\\\': 11544, \\\'git_sha1\\\': \\\'6f9739bb281a3d2ade8fb5f6dd6dde597a739c80\\\'},\\n    \\\'README.md\\\': {\\\'size\\\': 16885, \\\'git_sha1\\\': \\\'6ba98def472ad73c5cc6849d5da9e8389b0f88af\\\'},\\n    \\\'config.json\\\': {\\\'size\\\': 907, \\\'git_sha1\\\': \\\'f05252483c338f12a888a7f59698f1ea405e60e2\\\'},\\n    \\\'generation_config.json\\\': {\\\'size\\\': 239, \\\'git_sha1\\\': \\\'20a8a9156fc8c3f25295ca067f61fdf120d517c5\\\'},\\n    \\\'merges.txt\\\': {\\\'size\\\': 1671853, \\\'git_sha1\\\': \\\'31349551d90c7606f325fe0f11bbb8bd5fa0d7c7\\\'},\\n    \\\'model-00001-of-00002.safetensors\\\': {\\n        \\\'size\\\': 4988339832,\\n        \\\'sha256\\\': \\\'668eb0f1356638310db286f4819b223c12e3916934123f1a81b2b2c0e148c6a2\\\',\\n    },\\n    \\\'model-00002-of-00002.safetensors\\\': {\\n        \\\'size\\\': 4988350408,\\n        \\\'sha256\\\': \\\'c3c1625df80fe01211038bfa520629ebde6adf776556aa80cd49696d986d6657\\\',\\n    },\\n    \\\'model.safetensors.index.json\\\': {\\\'size\\\': 82253, \\\'git_sha1\\\': \\\'2ff701afad2fc062c13976386e06b7ade7d81f12\\\'},\\n    \\\'tokenizer.json\\\': {\\n        \\\'size\\\': 11422654,\\n        \\\'sha256\\\': \\\'aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4\\\',\\n    },\\n    \\\'tokenizer_config.json\\\': {\\\'size\\\': 9732, \\\'git_sha1\\\': \\\'417d038a63fa3de29cfde265caedae14d1a58d92\\\'},\\n    \\\'vocab.json\\\': {\\\'size\\\': 2776833, \\\'git_sha1\\\': \\\'4783fe10ac3adce15ac8f358ef5462739852c569\\\'},\\n}\\n\\ndef _qwen_verify_model_file(path, expected):\\n    if not path.is_file() or path.stat().st_size != expected[\\\'size\\\']:\\n        raise RuntimeError(\\\'Kaggle Qwen model input is missing a file or has an unexpected size.\\\')\\n    if \\\'sha256\\\' in expected:\\n        digest = hashlib.sha256()\\n        with path.open(\\\'rb\\\') as stream:\\n            for block in iter(lambda: stream.read(8 * 1024 * 1024), b\\\'\\\'):\\n                digest.update(block)\\n                _qwen_health.progress(phase=\\\'verify_model\\\', progress_unit=\\\'bytes\\\', progress_done=stream.tell(), progress_total=expected[\\\'size\\\'])\\n        actual = digest.hexdigest()\\n        wanted = expected[\\\'sha256\\\']\\n    else:\\n        digest = hashlib.sha1()\\n        digest.update(f"blob {expected[\\\'size\\\']}\\\\0".encode(\\\'ascii\\\'))\\n        with path.open(\\\'rb\\\') as stream:\\n            for block in iter(lambda: stream.read(8 * 1024 * 1024), b\\\'\\\'):\\n                digest.update(block)\\n                _qwen_health.progress(phase=\\\'verify_model\\\', progress_unit=\\\'bytes\\\', progress_done=stream.tell(), progress_total=expected[\\\'size\\\'])\\n        actual = digest.hexdigest()\\n        wanted = expected[\\\'git_sha1\\\']\\n    if actual != wanted:\\n        raise RuntimeError(\\\'Kaggle Qwen model input checksum does not match the pinned official revision.\\\')\\n\\nqwen_model_roots = set()\\nfor config_path in KAGGLE_INPUT_ROOT.rglob(\\\'config.json\\\'):\\n    candidate = config_path.parent\\n    if all((candidate / name).is_file() for name in QWEN_MODEL_FILES):\\n        qwen_model_roots.add(candidate.resolve())\\nif len(qwen_model_roots) != 1:\\n    raise FileNotFoundError(\\n        f\\\'Expected exactly one Kaggle Input for the pinned Qwen model; found {len(qwen_model_roots)}.\\\'\\n    )\\nQWEN_LOCAL_MODEL_DIR = next(iter(qwen_model_roots))\\n_qwen_health.set_phase(\\\'verify_model\\\')\\nfor filename, expected in QWEN_MODEL_FILES.items():\\n    _qwen_verify_model_file(QWEN_LOCAL_MODEL_DIR / filename, expected)\\n\\nqwen_config = json.loads((QWEN_LOCAL_MODEL_DIR / \\\'config.json\\\').read_text(encoding=\\\'utf-8\\\'))\\nif qwen_config.get(\\\'model_type\\\') != \\\'qwen3\\\' or (qwen_config.get(\\\'quantization_config\\\') or {}).get(\\\'quant_method\\\') != \\\'awq\\\':\\n    raise RuntimeError(\\\'Kaggle model input is not the expected Qwen3 AWQ checkpoint.\\\')\\nqwen_index = json.loads((QWEN_LOCAL_MODEL_DIR / \\\'model.safetensors.index.json\\\').read_text(encoding=\\\'utf-8\\\'))\\nif set(qwen_index.get(\\\'weight_map\\\', {}).values()) != {\\n    \\\'model-00001-of-00002.safetensors\\\', \\\'model-00002-of-00002.safetensors\\\'\\n}:\\n    raise RuntimeError(\\\'Kaggle model input has an unexpected safetensors shard map.\\\')\\n\\ndef _qwen_from_local_snapshot(original_loader):\\n    def _wrapped_loader(cls, model_name_or_path, *args, **kwargs):\\n        if str(model_name_or_path) == QWEN_MODEL_REPO:\\n            kwargs.pop(\\\'revision\\\', None)\\n            kwargs.pop(\\\'token\\\', None)\\n            kwargs[\\\'local_files_only\\\'] = True\\n            os.environ[\\\'HF_HUB_OFFLINE\\\'] = \\\'1\\\'\\n            os.environ[\\\'TRANSFORMERS_OFFLINE\\\'] = \\\'1\\\'\\n            return original_loader(str(QWEN_LOCAL_MODEL_DIR), *args, **kwargs)\\n        return original_loader(model_name_or_path, *args, **kwargs)\\n    return classmethod(_wrapped_loader)\\n\\n_qwen_original_tokenizer_loader = AutoTokenizer.from_pretrained\\n_qwen_original_model_loader = AutoModelForCausalLM.from_pretrained\\nAutoTokenizer.from_pretrained = _qwen_from_local_snapshot(_qwen_original_tokenizer_loader)\\nAutoModelForCausalLM.from_pretrained = _qwen_from_local_snapshot(_qwen_original_model_loader)\\n\\ndef _qwen_local_revision_lock(hf_token):\\n    locked = {}\\n    for key, model in pipeline.MODELS.items():\\n        if model[\\\'backend\\\'] == \\\'google_gemini_api\\\':\\n            revision = model[\\\'api_model_id\\\']\\n        elif key == \\\'qwen3_14b_awq\\\':\\n            revision = QWEN_MODEL_REVISION\\n        else:\\n            raise RuntimeError(\\\'Unexpected Hugging Face model in the Kaggle Qwen lock.\\\')\\n        locked[key] = {\\n            \\\'model_id\\\': model[\\\'model_id\\\'],\\n            \\\'backend\\\': model[\\\'backend\\\'],\\n            \\\'provider\\\': model[\\\'provider\\\'],\\n            \\\'revision\\\': revision,\\n            \\\'template_version\\\': model[\\\'template_version\\\'],\\n        }\\n    return locked\\n\\npipeline._model_revision_lock = _qwen_local_revision_lock\\nprint(\\\'Verified all 12 files for the pinned Qwen3-14B-AWQ snapshot; model loading will use the local Kaggle Input.\\\', flush=True)\\n\\nsmoke_context = pipeline.prepare_pipeline(\\n    mode=\\\'smoke\\\', model_key=\\\'qwen3_14b_awq\\\', data_path=SOURCE, workdir=WORKDIR,\\n    hf_token=None, qwen_batch_size=QWEN_BATCH_SIZE\\n)\\nattach_durable_cache(smoke_context, enable_control=True)\\n_migrate_legacy_qwen_cache(smoke_context)\\nsmoke_result = pipeline.run_stage(\\n    smoke_context, batch_size=QWEN_BATCH_SIZE, allow_qwen_hardware_change=True\\n)\\n# Kaggle uses the reviewed bundle attached as a read-only Dataset.\\npersist_qwen_cache(LOCAL_CACHE)\\nimport gc\\nimport torch\\ngc.collect()\\nif torch.cuda.is_available():\\n    torch.cuda.synchronize()\\n    torch.cuda.empty_cache()\\n    free_bytes, total_bytes = torch.cuda.mem_get_info()\\n    print(f\\\'VRAM trống sau smoke: {free_bytes / 1024**3:.2f}/{total_bytes / 1024**3:.2f} GiB\\\')\\nprint(\\\'Preflight:\\\', json.dumps({k: v for k, v in smoke_result.items() if k not in {\\\'qa\\\', \\\'cache\\\'}}, ensure_ascii=False, indent=2, default=str))\\nsmoke_completion = float(smoke_result.get(\\\'completion_rate\\\', 0) or 0)\\nif smoke_completion < 0.99:\\n    raise RuntimeError(\\n        f\\\'Preflight chỉ hoàn tất {smoke_completion:.1%}; đã lưu checkpoint trong /kaggle/working và dừng trước full. \\\'\\n        \\\'Không chạy full với đầu ra đang lỗi/lặp. Hãy dùng bundle đã sửa.\\\'\\n    )\\nif smoke_result.get(\\\'status\\\') == \\\'quality_gate_failed\\\':\\n    print(\\\'Preflight đủ đầu ra nhưng còn cờ QA kỹ thuật; theo lựa chọn của bạn, sẽ lưu cờ để rà soát sau full.\\\')\\nsmoke_generation = smoke_result.get(\\\'generation\\\', {})\\nsmoke_new = int(smoke_generation.get(\\\'new_completed\\\', 0))\\nsmoke_seconds = float(smoke_generation.get(\\\'elapsed_seconds\\\', 0) or 0)\\nif smoke_new and smoke_seconds > 0:\\n    rate = smoke_new / smoke_seconds\\n    eta_hours = pipeline.EXPECTED_UNIQUE_TITLES / rate / 3600\\n    print(f\\\'Tốc độ preflight: {rate * 3600:.1f} title/giờ; ETA thô cho full: {eta_hours:.1f} giờ (chưa tính reset/tải model lại).\\\')\\n\\nfull_result = None\\nfor resume_round in range(1, 4):\\n    full_context = pipeline.prepare_pipeline(\\n        mode=\\\'full\\\', model_key=\\\'qwen3_14b_awq\\\', data_path=SOURCE, workdir=WORKDIR,\\n        hf_token=None, qwen_batch_size=QWEN_BATCH_SIZE\\n    )\\n    attach_durable_cache(full_context, enable_control=True)\\n    full_result = pipeline.run_stage(\\n        full_context,\\n        batch_size=QWEN_FULL_BATCH_SIZE,\\n        allow_qwen_batch_change=True,\\n        full_titles_per_session=pipeline.EXPECTED_UNIQUE_TITLES,\\n        allow_unvalidated_full=True,\\n        defer_full_quality_review=True,\\n        allow_qwen_hardware_change=True,\\n    )\\n    persist_qwen_cache(LOCAL_CACHE)\\n    if full_result.get(\\\'status\\\') == \\\'reload_requested\\\':\\n        raise RuntimeError(\\\'Kaggle does not hot-reload bundles. Export the checkpoint, review a new notebook/bundle, then resume.\\\')\\n    if full_result.get(\\\'status\\\') != \\\'checkpoint_saved_continue\\\':\\n        break\\n    if resume_round < 3:\\n        print(f"Còn title chưa hoàn tất; sẽ thử lại lượt {resume_round + 1}/3 từ checkpoint.")\\n    else:\\n        print(\\\'Vẫn còn title chưa hoàn tất sau 3 lượt; checkpoint còn trong /kaggle/working. Chạy lại notebook để thử tiếp nếu Files persistence hoặc snapshot ZIP còn sẵn.\\\')\\nassert full_result is not None\\nprint(\\\'Full run:\\\', json.dumps({k: v for k, v in full_result.items() if k not in {\\\'qa\\\', \\\'cache\\\'}}, ensure_ascii=False, indent=2, default=str))\\nif full_result.get(\\\'status\\\') != \\\'all_titles_complete\\\':\\n    print(\\\'Chưa xong toàn bộ; checkpoint đã lưu trong /kaggle/working. Sau khi nối runtime lại, chọn Runtime → Run all để resume.\\\')\\n\\n_qwen_health.cell_finished(final=False)\\n\\n\\n\\n_qwen_health.set_phase(\\\'review_qa\\\', cell=9)\\nimport json\\nimport hashlib\\nimport pandas as pd\\n\\nstate_path = WORKDIR / \\\'hybrid_run_state.json\\\'\\nstate = json.loads(state_path.read_text(encoding=\\\'utf-8\\\'))\\nfull_status = state.get(\\\'models\\\', {}).get(\\\'qwen3_14b_awq\\\', {}).get(\\\'stages\\\', {}).get(\\\'full\\\', {}).get(\\\'status\\\')\\nif full_status != \\\'all_titles_complete\\\':\\n    raise RuntimeError(f\\\'Qwen full chưa hoàn tất (status={full_status}); resume full trước khi rà soát.\\\')\\n\\nOUTPUT = WORKDIR / \\\'bilingual_zh_vi_cado_title_qwen3_14b_awq.parquet\\\'\\nif not OUTPUT.exists():\\n    raise FileNotFoundError(f\\\'Không thấy Parquet hoàn chỉnh: {OUTPUT}\\\')\\nresult_frame = pd.read_parquet(OUTPUT)\\nif len(result_frame) != pipeline.EXPECTED_ROWS:\\n    raise AssertionError(f\\\'Số dòng sai: {len(result_frame)}; cần {pipeline.EXPECTED_ROWS}.\\\')\\nqa_status_col = \\\'title_vi_qwen3_14b_awq_qa_status\\\'\\nhard_flags_col = \\\'title_vi_qwen3_14b_awq_qa_flags\\\'\\nadvisory_col = \\\'title_vi_qwen3_14b_awq_advisory_flags\\\'\\nsemantic_col = \\\'title_vi_qwen3_14b_awq_semantic_verified\\\'\\nrequired = [qa_status_col, hard_flags_col, advisory_col, semantic_col, \\\'title_vi_candidate_qwen3_14b_awq\\\']\\nmissing = [name for name in required if name not in result_frame.columns]\\nif missing:\\n    raise AssertionError(f\\\'Thiếu cột QA/output: {missing}\\\')\\n\\nunique_review = result_frame.drop_duplicates(\\\'title_zh\\\').copy()\\nreview_cols = [name for name in [\\n    \\\'product_id\\\', \\\'title_zh\\\', \\\'title_vi\\\', \\\'title_vi_cleaned\\\',\\n    \\\'title_vi_candidate_qwen3_14b_awq\\\', qa_status_col, hard_flags_col, advisory_col,\\n] if name in unique_review.columns]\\nall_review_path = WORKDIR / \\\'qwen3_14b_awq_postrun_qa_all_unique_titles.csv\\\'\\nunique_review[review_cols].to_csv(all_review_path, index=False, encoding=\\\'utf-8-sig\\\')\\nflagged = unique_review[\\n    unique_review[qa_status_col].ne(\\\'pass\\\')\\n    | unique_review[hard_flags_col].fillna(\\\'[]\\\').ne(\\\'[]\\\')\\n    | unique_review[advisory_col].fillna(\\\'[]\\\').ne(\\\'[]\\\')\\n]\\nflagged_path = WORKDIR / \\\'qwen3_14b_awq_postrun_qa_flagged_titles.csv\\\'\\nflagged[review_cols].to_csv(flagged_path, index=False, encoding=\\\'utf-8-sig\\\')\\n\\nstatus_counts = unique_review[qa_status_col].fillna(\\\'missing\\\').value_counts().to_dict()\\nhard_counts = {}\\nfor value in unique_review[hard_flags_col].fillna(\\\'[]\\\'):\\n    for flag in json.loads(value):\\n        hard_counts[flag] = hard_counts.get(flag, 0) + 1\\nsummary = {\\n    \\\'rows\\\': len(result_frame),\\n    \\\'unique_titles\\\': int(unique_review[\\\'title_zh\\\'].nunique()),\\n    \\\'qa_status_counts_unique_titles\\\': status_counts,\\n    \\\'hard_flag_counts_unique_titles\\\': hard_counts,\\n    \\\'flagged_unique_titles\\\': len(flagged),\\n    \\\'semantic_verified\\\': bool(result_frame[semantic_col].fillna(True).astype(bool).any()),\\n    \\\'parquet_sha256\\\': pipeline.quality.sha256_file(OUTPUT),\\n}\\nsummary_path = WORKDIR / \\\'qwen3_14b_awq_postrun_qa_summary.json\\\'\\nsummary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding=\\\'utf-8\\\')\\nsummary_md = [\\n    \\\'# Qwen3-14B-AWQ (4-bit AWQ) title — QA sau full run\\\', \\\'\\\',\\n    f"- Số dòng: {summary[\\\'rows\\\']:,}",\\n    f"- Title duy nhất: {summary[\\\'unique_titles\\\']:,}",\\n    f"- Trạng thái QA: `{json.dumps(status_counts, ensure_ascii=False)}`",\\n    f"- Title có cờ cần xem: {summary[\\\'flagged_unique_titles\\\']:,}",\\n    f"- SHA-256 Parquet: `{summary[\\\'parquet_sha256\\\']}`",\\n    \\\'- `semantic_verified=False`: chưa có người xác nhận ngữ nghĩa.\\\',\\n    \\\'\\\',\\n    f\\\'- Danh sách đầy đủ: `{all_review_path.name}`\\\',\\n    f\\\'- Danh sách cần xem: `{flagged_path.name}`\\\',\\n]\\n(WORKDIR / \\\'qwen3_14b_awq_postrun_qa_summary.md\\\').write_text(\\\'\\\\n\\\'.join(summary_md) + \\\'\\\\n\\\', encoding=\\\'utf-8\\\')\\nprint(json.dumps(summary, ensure_ascii=False, indent=2))\\n\\n_qwen_health.cell_finished(final=False)\\n\\n\\n\\n_qwen_health.set_phase(\\\'export_checkpoint\\\', cell=10)\\n# Xuất checkpoint để tải hoặc đưa vào private Kaggle Dataset phiên sau.\\nimport zipfile\\nfrom pathlib import Path\\nfrom IPython.display import FileLink, display\\n\\ncheckpoint_files = [\\n    \\\'cache_qwen3_14b_awq.sqlite3\\\',\\n    \\\'hybrid_environment_lock.json\\\',\\n    \\\'hybrid_run_state.json\\\',\\n    \\\'legacy_cache_import.json\\\',\\n    \\\'qwen3_14b_awq_kaggle_package_lock.json\\\',\\n    \\\'qwen_live_status.json\\\',\\n    \\\'qwen_health.json\\\',\\n]\\ncheckpoint_path = Path(\\\'/kaggle/working/qwen3_14b_awq_kaggle_checkpoint.zip\\\')\\nwith zipfile.ZipFile(checkpoint_path, \\\'w\\\', compression=zipfile.ZIP_DEFLATED) as archive:\\n    for name in checkpoint_files:\\n        path = WORKDIR / name\\n        if path.is_file():\\n            archive.write(path, arcname=name)\\nprint(\\\'Checkpoint ZIP:\\\', checkpoint_path)\\nprint(\\\'Nếu chưa bật Files persistence: tải ZIP này trước khi đóng phiên; để resume, thêm ZIP vào private Kaggle Dataset đầu vào mới.\\\')\\ndisplay(FileLink(str(checkpoint_path)))\\n\\n_qwen_health.cell_finished(final=True)\\n\', "qwen_fresh_worker.py", "exec"), globals())\nexcept BaseException as _worker_error:\n    if "_qwen_health" in globals():\n        _qwen_health.fail(_worker_error)\n    print("[qwen-worker-error] " + _worker_json.dumps({"error_class": type(_worker_error).__name__}), flush=True)\n    raise SystemExit(1) from None\n', encoding='utf-8')
print('[qwen-fresh-worker] starting clean interpreter after dependency installation', flush=True)
_worker = _worker_subprocess.Popen([_worker_sys.executable, '-u', str(_worker_path)],
    stdout=_worker_subprocess.PIPE, stderr=_worker_subprocess.STDOUT,
    text=True, encoding='utf-8', errors='replace', bufsize=1)
with (WORKDIR / 'qwen_worker.stdout.log').open('a', encoding='utf-8') as _worker_log:
    for _worker_line in _worker.stdout:
        print(_worker_line, end='', flush=True)
        _worker_log.write(_worker_line)
        _worker_log.flush()
_worker_exit = _worker.wait()
if _worker_exit:
    raise RuntimeError('Qwen worker stopped; inspect qwen_health.json and safe diagnostics.')
print('[qwen-fresh-worker] finished; output and checkpoint are under /kaggle/working', flush=True)
